# 2단계. 적용 조건 — 전제·실패 신호, 대안 비교

**런타임: CPU.** 1단계 노트북과 같은 런타임이면 `results/`, `base.pt` 캐시를 재사용합니다.

In [ ]:
import os, sys, time, dataclasses
if not os.path.exists("fg8.py"):
    !git clone -q https://github.com/HisameOgasahara/tml_p1_rl.git
    %cd tml_p1_rl
import numpy as np, torch, matplotlib.pyplot as plt
import fg8
torch.set_num_threads(os.cpu_count())
print("torch", torch.__version__, "| threads", torch.get_num_threads())
base = fg8.get_base()          # 처음 한 번 RF 사전학습(~1분), 이후 base.pt 재사용

In [ ]:
SEEDS = [0, 1, 2]

def run_all(cfgs):
    out = {}
    for name, c in cfgs.items():
        t0 = time.time()
        out[name] = [fg8.run(base, dataclasses.replace(c, seed=s), name, verbose=(s == 0)) for s in SEEDS]
        print(f"{name}: {time.time() - t0:.0f}s")
    return out

def final(runs, key):
    return np.array([r["ode"][-1][key] for r in runs])

def table(runs_by_name, keys=("reward", "target_mass", "off_support", "within_std")):
    print(f"{'run':<16}" + "".join(f"{k:>16}" for k in keys) + f"{'train s':>10}{'NFE':>12}")
    base_m = next(iter(runs_by_name.values()))[0]["ode"][0]
    print(f"{'base':<16}" + "".join(f"{base_m[k]:>16.4f}" for k in keys))
    for name, runs in runs_by_name.items():
        cells = "".join(f"{final(runs, k).mean():>9.4f}±{final(runs, k).std():<6.4f}" for k in keys)
        print(f"{name:<16}{cells}{np.mean([r['time'][-1] for r in runs]):>10.1f}{runs[0]['nfe'][-1]:>12,}")

def curves(runs_by_name, x="it", key="reward", ax=None):
    ax = ax or plt.gca()
    for name, runs in runs_by_name.items():
        xs = np.array(runs[0][x]); ys = np.array([[m[key] for m in r["ode"]] for r in runs])
        ax.plot(xs, ys.mean(0), label=name); ax.fill_between(xs, ys.min(0), ys.max(0), alpha=0.2)
    ax.set_xlabel(x); ax.set_ylabel(f"ODE {key}"); ax.legend()

def scatter(ax, pts, title):
    pts = np.asarray(pts); ax.scatter(pts[:, 0], pts[:, 1], s=2)
    m = fg8.MODES.numpy(); ax.scatter(m[list(fg8.TARGET), 0], m[list(fg8.TARGET), 1], marker="x", c="r")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(title)

## 2-① 언제 이식 가능한가 — 전제, 실패, 신호 (§8, A.3, Q24)

| 전제 | 신호 |
|---|---|
| 그룹 안 보상에 분산이 있다 | std=0 그룹 비율 ≈ p^G + (1-p)^G, 보상 정체 |
| 학습 SDE와 평가 ODE 분포가 같다 | 같은 policy의 SDE(T=10) vs ODE(T=40) reward 차이 |
| clip이 모든 t에서 작동한다 | t 구간별 clip 비율 (GRPO-Guard) |
| 보상 상승 = 품질 상승 | reward ↑ 동시에 off-support ↑ / within-std ↓ |

In [ ]:
dense = run_all({"grpo_kl0.04": fg8.Cfg(), "grpo_kl0": fg8.Cfg(beta_kl=0.0)})
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for name, rs in dense.items():
    r = rs[0]
    ax[0].plot(r["it"], [m["reward"] - s for m, s in zip(r["ode"], r["sde_reward"])], label=name)
ax[0].set_title("ODE reward - SDE reward (seed 0)"); ax[0].legend()
for name, rs in dense.items():
    r = rs[0]
    ax[1].plot([m["reward"] for m in r["ode"]], [m["within_std"] for m in r["ode"]], "o-", label=name)
ax[1].axhline(fg8.SIGMA, ls="--", c="k"); ax[1].set_xlabel("ODE reward"); ax[1].set_ylabel("within-std"); ax[1].legend()
ax[1].set_title("reward vs diversity (점선 = 데이터 σ)")
im = ax[2].imshow(np.array(dense["grpo_kl0.04"][0]["clip_by_t"]).T, aspect="auto", origin="lower")
ax[2].set_xlabel("iteration"); ax[2].set_ylabel("step k (0 = t=1 noise)"); ax[2].set_title("clip fraction by t (kl0.04, seed 0)")
plt.colorbar(im, ax=ax[2]); plt.show()

### 그룹 분산 전제를 깨기: 희소 보상 (mode 0 반경 σ 안이면 1)

먼저 base에서 std=0 그룹 비율이 이론값 p^G + (1-p)^G와 맞는지 확인하고, 총 샘플 수(batch=384)를 고정한 채 G=4 vs G=24로 학습합니다.

In [ ]:
zs = fg8.check_zero_std(base)
print("PASS zero-std theory:", zs)
sparse = run_all({
    "sparse_G4":  fg8.Cfg(reward="sparse", G=4),
    "sparse_G24": fg8.Cfg(reward="sparse", G=24),
})
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
curves(sparse, key="reward", ax=ax[0]); ax[0].set_ylabel("ODE hit rate")
for name, rs in sparse.items():
    ax[1].plot(np.mean([r["zero_std"] for r in rs], 0), label=name)
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("std=0 group fraction"); ax[1].legend()
plt.show()

## 2-② 대안 중 언제 고르고 언제 피하나

같은 base·보상·iteration당 샘플 수(384)로 비교.

- **(a) PPO**: SDE rollout·ratio·clip·KL은 Flow-GRPO와 동일, advantage만 critic V(x_t, t)로. 2D에서는 critic 학습이 쉬워 PPO에 유리할 수 있음.
- **(b) DGPO, DiffusionNFT**: 튜토리얼 Q25 기준(정답) — SDE 비용을 낼 수 있고 스텝별 likelihood가 필요하면 Flow-GRPO, ODE만 쓰거나 저장·속도가 중요하면 DGPO/NFT. toy 결과가 이 기준과 맞는지 확인.
- DGPO `dgpo_beta_T`, NFT `nft_beta`는 손실 스케일 규약에 묶인 값(§4.8, §5.7)이라 toy에서 조정이 필요할 수 있음.

In [ ]:
alts = run_all({
    "grpo_kl0.04": fg8.Cfg(),
    "ppo":        fg8.Cfg(algo="ppo"),
    "dgpo":       fg8.Cfg(algo="dgpo"),
    "nft":        fg8.Cfg(algo="nft"),
})
ppo_sparse = run_all({"ppo_sparse": fg8.Cfg(algo="ppo", reward="sparse")})

In [ ]:
table(alts)
print()
for name, rs in alts.items():
    print(f"{name:<6} rollout buffer floats: {rs[0]['buffer_floats']:,}")
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
curves(alts, x="it", ax=ax[0]); curves(alts, x="nfe", ax=ax[1]); curves(alts, x="time", ax=ax[2])
plt.show()
fig, ax = plt.subplots(1, 4, figsize=(17, 4))
for a_, (name, rs) in zip(ax, alts.items()):
    scatter(a_, rs[0]["final_samples"], f"{name} (seed 0)")
plt.show()
print("sparse reward: Flow-GRPO G=4 / G=24 vs PPO")
table({**sparse, **ppo_sparse}, keys=("reward",))